In [45]:
import numpy as np
import pandas as pd
from psycopg.types import none

meta = pd.read_excel("../data/raw/gios/Metadane oraz kody stacji i stanowisk pomiarowych.xlsx")
print(meta.columns.tolist())
meta.head()

['Nr', 'Kod stacji', 'Kod międzynarodowy', 'Nazwa stacji', 'Stary Kod stacji \n(o ile inny od aktualnego)', 'Data uruchomienia', 'Data zamknięcia', 'Typ stacji', 'Typ obszaru', 'Rodzaj stacji', 'Województwo', 'Miejscowość', 'Adres', 'WGS84 φ N', 'WGS84 λ E']


,Nr,Kod stacji,Kod międzynarodowy,Nazwa stacji,Stary Kod stacji \n(o ile inny od aktualnego),Data uruchomienia,Data zamknięcia,Typ stacji,Typ obszaru,Rodzaj stacji,Województwo,Miejscowość,Adres,WGS84 φ N,WGS84 λ E
0,1,DsBialka,NaN,Białka,NaN,1990-01-03,2005-12-31,przemysłowa,podmiejski,kontenerowa stacjonarna,DOLNOŚLĄSKIE,Białka,NaN,51.197783,16.117390
1,2,DsBielGrot,NaN,Bielawa - ul. Grota Roweckiego,NaN,1994-01-02,2003-12-31,tło,miejski,w budynku,DOLNOŚLĄSKIE,Bielawa,ul. Grota Roweckiego 6,50.682510,16.617348
2,3,DsBogatFrancMOB,PL0602A,Bogatynia Mobil,DsBogatMob,2015-01-01,2015-12-31,tło,miejski,mobilna,DOLNOŚLĄSKIE,Bogatynia,ul. Francuska/Kręta,50.940998,14.916790
3,4,DsBogChop,PL0315A,Bogatynia - Chopina,NaN,1996-01-01,2013-12-31,przemysłowa,miejski,kontenerowa stacjonarna,DOLNOŚLĄSKIE,Bogatynia,ul. Chopina 35,50.905856,14.967175
4,5,DsBogZatonieMob,PL0576A,Bogatynia - Mobil,NaN,2012-01-01,2012-12-31,przemysłowa,miejski,mobilna,DOLNOŚLĄSKIE,Bogatynia,"ul. Konrada, Zatonie",50.943245,14.913327


In [46]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.gios_archive import load_gios_hourly

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
miasta = ["Kraków","Warszawa", "Katowice", "Białystok", "Olsztyn","Łódź", "Lublin", "Gdańsk", "Bydgoszcz", "Rzeszów","Szczecin"]

wynik = meta[meta["Miejscowość"].isin(miasta) & (meta["Typ obszaru"] == 'miejski') & (meta['Typ stacji'] == 'tło')]


wynik.sort_values("Miejscowość")

,Nr,Kod stacji,Kod międzynarodowy,Nazwa stacji,Stary Kod stacji \n(o ile inny od aktualnego),Data uruchomienia,Data zamknięcia,Typ stacji,Typ obszaru,Rodzaj stacji,Województwo,Miejscowość,Adres,WGS84 φ N,WGS84 λ E
695,696,PdBialUpalna,PL0841A,"Białystok, ul. Upalna",PdBialWaszyn,2024-01-25,NaT,tło,miejski,kontenerowa stacjonarna,PODLASKIE,Białystok,ul. Upalna 26,53.129601,23.108054
696,697,PdBialWarsza,PL0496A,"Białystok, ul. Warszawska",PdBialyWarszawska,2010-01-01,NaT,tło,miejski,kontenerowa stacjonarna,PODLASKIE,Białystok,ul. Warszawska 75 A,53.129306,23.181744
697,698,PdBialWaszyn,PL0148A,"Białystok, ul. Waszyngtona",PdBialyMiejska,2004-09-01,2023-11-13,tło,miejski,kontenerowa stacjonarna,PODLASKIE,Białystok,ul. Waszyngtona 16,53.126689,23.155869
698,699,PdBialyBroniewsk,NaN,Bialystok ul. Broniewskiego 1,NaN,1994-01-01,2002-12-31,tło,miejski,w budynku,PODLASKIE,Białystok,ul. Broniewskiego 1,53.140469,23.134272
699,700,PdBialyLegionowa,NaN,Białystok ul Legionowa 8,NaN,1998-01-01,2003-12-31,tło,miejski,w budynku,PODLASKIE,Białystok,ul. Legionowa 8,53.130933,23.159853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,345,LdLodzWSSEMRokicin,NaN,Łódź-Rokicińska144,NaN,1971-01-02,2003-12-31,tło,miejski,w budynku,ŁÓDZKIE,Łódź,ul. Rokicińska 144,51.758056,19.560278
345,346,LdLodzWSSEMWici3,PL0037A,Łódź-Wici3,NaN,1994-01-07,2010-01-01,tło,miejski,w budynku,ŁÓDZKIE,Łódź,ul. Wici 3,51.798611,19.384722
346,347,LdLodzWSSEMWilensk,PL0369A,Łódź-Wileńska25,NaN,1994-01-01,2010-01-01,tło,miejski,w budynku,ŁÓDZKIE,Łódź,ul. Wileńska 25,51.752778,19.428333
347,348,LdLodzWSSEMWodna40,PL0002A,Łódź-WSSE Wodna40,NaN,1998-12-25,2010-01-01,tło,miejski,w budynku,ŁÓDZKIE,Łódź,ul. Wodna 40,51.761667,19.476944


In [48]:
start = pd.to_datetime(wynik['Data uruchomienia'])
end = pd.to_datetime(wynik['Data zamknięcia']).fillna(pd.Timestamp("2100-01-01"))

dzialajace = wynik[(start <= "2015-01-01") & (end >= "2025-01-01")]
dzialajace.sort_values("Miejscowość")

,Nr,Kod stacji,Kod międzynarodowy,Nazwa stacji,Stary Kod stacji \n(o ile inny od aktualnego),Data uruchomienia,Data zamknięcia,Typ stacji,Typ obszaru,Rodzaj stacji,Województwo,Miejscowość,Adres,WGS84 φ N,WGS84 λ E
696,697,PdBialWarsza,PL0496A,"Białystok, ul. Warszawska",PdBialyWarszawska,2010-01-01,NaT,tło,miejski,kontenerowa stacjonarna,PODLASKIE,Białystok,ul. Warszawska 75 A,53.129306,23.181744
182,183,KpBydWarszaw,PL0064A,"Bydgoszcz, ul. Warszawska",KpBydgWarszawska,1995-01-21,NaT,tło,miejski,kontenerowa stacjonarna,KUJAWSKO-POMORSKIE,Bydgoszcz,ul. Warszawska 10,53.134083,17.995708
834,835,PmGdaLeczkow,PL0052A,"Gdańsk, ul. Leczkowa","Pm.a08a, PmGdaLeczk08, PmGdaLecz08m",1998-10-01,NaT,tło,miejski,kontenerowa stacjonarna,POMORSKIE,Gdańsk,ul. Leczkowa,54.380279,18.620274
836,837,PmGdaPowWars,PL0045A,"Gdańsk, ul. Powstańców Warszawskich","Pm.a01a, PmGdaPoWie01",1996-10-01,NaT,tło,miejski,kontenerowa stacjonarna,POMORSKIE,Gdańsk,ul. Powstańców Warszawskich,54.353336,18.635283
837,838,PmGdaPowWiel,PL0497A,"Gdańsk, ul. Powstańców Wielkopolskich",Pm.01w.02m,2010-01-01,NaT,tło,miejski,wolnostojący pobornik pyłu,POMORSKIE,Gdańsk,ul. Powstańców Wielkopolskich,54.398639,18.614333
838,839,PmGdaWyzwole,PL0047A,"Gdańsk, ul. Wyzwolenia","Pm.a03a, PmGdaWyzwo03",1998-09-01,NaT,tło,miejski,kontenerowa stacjonarna,POMORSKIE,Gdańsk,ul. Wyzwolenia,54.400833,18.657497
941,942,SlKatoKossut,PL0008A,"Katowice, ul. Kossutha",SlKatowKato_kossu,1992-01-01,NaT,tło,miejski,kontenerowa stacjonarna,ŚLĄSKIE,Katowice,ul. Kossutha 6,50.264611,18.975028
453,454,MpKrakBujaka,PL0501A,"Kraków, ul. Bujaka",MpKrakowWIOSBuja6119,2010-01-01,NaT,tło,miejski,kontenerowa stacjonarna,MAŁOPOLSKIE,Kraków,ul. Bujaka,50.010575,19.949189
297,298,LbLubObywate,PL0507A,"Lublin, ul. Obywatelska",LbLublinWIOS,2010-05-21,NaT,tło,miejski,kontenerowa stacjonarna,LUBELSKIE,Lublin,ul. Obywatelska 13,51.259431,22.569133
298,299,LbLubSliwins,PL0085A,"Lublin, ul. Śliwińskiego",LbLublin_Sliwins_5,2001-07-01,NaT,tło,miejski,kontenerowa stacjonarna,LUBELSKIE,Lublin,ul. Śliwińskiego 5,51.273078,22.551675


In [49]:
xls = pd.ExcelFile('../data/raw/gios/Metadane oraz kody stacji i stanowisk pomiarowych.xlsx')
print(xls.sheet_names)

['STACJE', 'STANOWISKA']


In [50]:
stanowiska = pd.read_excel(xls, sheet_name="STANOWISKA")
print(stanowiska.columns.tolist())
stanowiska.sample(5)

['Nr', 'Kod stanowiska', 'Kod stacji', 'Nazwa stacji', 'Wskaźnik - kod', 'Wskaźnik', 'Czas uśredniania', 'Typ pomiaru', 'Data uruchomienia', 'Data zamknięcia', 'Województwo', 'Nazwa strefy']


,Nr,Kod stanowiska,Kod stacji,Nazwa stacji,Wskaźnik - kod,Wskaźnik,Czas uśredniania,Typ pomiaru,Data uruchomienia,Data zamknięcia,Województwo,Nazwa strefy
1659,1660,LbTomaLubPSPMOB-PM10-24g,LbTomaLubPSPMOB,"Tomaszów Lubelski, ul. Janusza Petera",PM10,pył zawieszony PM10,24-godzinny,manualny,2025-01-08 00:00:00,NaN,LUBELSKIE,strefa lubelska
3324,3325,OpKKozBSmial-SO2-1g,OpKKozBSmial,"Kędzierzyn-Koźle, ul. Śmiałego",SO2,dwutlenek siarki,1-godzinny,automatyczny,2005-01-01 00:00:00,NaN,OPOLSKIE,strefa opolska
5358,5359,WpKaliSawick-NOx-1g,WpKaliSawick,"Kalisz, ul. Wyszyńskiego",NOx,tlenki azotu,1-godzinny,automatyczny,2010-01-01 00:00:00,NaN,WIELKOPOLSKIE,miasto Kalisz
1416,1417,KpZielBoryTu-PM10-24g,KpZielBoryTu,"Zielonka, Bory Tucholskie",PM10,pył zawieszony PM10,24-godzinny,manualny,2010-01-01 00:00:00,NaN,KUJAWSKO-POMORSKIE,strefa kujawsko - pomorska
4632,4633,SkSkarz1MajaMOB-PM10-24g,SkSkarz1MajaMOB,"Skarżysko-Kamienna, ul. 1 Maja",PM10,pył zawieszony PM10,24-godzinny,manualny,2024-01-04 00:00:00,2025-12-29 00:00:00,ŚWIĘTOKRZYSKIE,strefa świętokrzyska


In [51]:
pm25 = stanowiska[stanowiska['Wskaźnik - kod'] == 'PM2.5']

kody_pm25 = pm25['Kod stacji'].unique()

In [52]:
pm10 = stanowiska[stanowiska['Wskaźnik - kod'] == 'PM10']

kody_pm10 = pm10['Kod stacji'].unique()

stacje_pm10 = dzialajace[dzialajace['Kod stacji'].isin(kody_pm10)]
stacje_pm10.sort_values("Miejscowość")

,Nr,Kod stacji,Kod międzynarodowy,Nazwa stacji,Stary Kod stacji \n(o ile inny od aktualnego),Data uruchomienia,Data zamknięcia,Typ stacji,Typ obszaru,Rodzaj stacji,Województwo,Miejscowość,Adres,WGS84 φ N,WGS84 λ E
696,697,PdBialWarsza,PL0496A,"Białystok, ul. Warszawska",PdBialyWarszawska,2010-01-01,NaT,tło,miejski,kontenerowa stacjonarna,PODLASKIE,Białystok,ul. Warszawska 75 A,53.129306,23.181744
182,183,KpBydWarszaw,PL0064A,"Bydgoszcz, ul. Warszawska",KpBydgWarszawska,1995-01-21,NaT,tło,miejski,kontenerowa stacjonarna,KUJAWSKO-POMORSKIE,Bydgoszcz,ul. Warszawska 10,53.134083,17.995708
834,835,PmGdaLeczkow,PL0052A,"Gdańsk, ul. Leczkowa","Pm.a08a, PmGdaLeczk08, PmGdaLecz08m",1998-10-01,NaT,tło,miejski,kontenerowa stacjonarna,POMORSKIE,Gdańsk,ul. Leczkowa,54.380279,18.620274
836,837,PmGdaPowWars,PL0045A,"Gdańsk, ul. Powstańców Warszawskich","Pm.a01a, PmGdaPoWie01",1996-10-01,NaT,tło,miejski,kontenerowa stacjonarna,POMORSKIE,Gdańsk,ul. Powstańców Warszawskich,54.353336,18.635283
838,839,PmGdaWyzwole,PL0047A,"Gdańsk, ul. Wyzwolenia","Pm.a03a, PmGdaWyzwo03",1998-09-01,NaT,tło,miejski,kontenerowa stacjonarna,POMORSKIE,Gdańsk,ul. Wyzwolenia,54.400833,18.657497
941,942,SlKatoKossut,PL0008A,"Katowice, ul. Kossutha",SlKatowKato_kossu,1992-01-01,NaT,tło,miejski,kontenerowa stacjonarna,ŚLĄSKIE,Katowice,ul. Kossutha 6,50.264611,18.975028
453,454,MpKrakBujaka,PL0501A,"Kraków, ul. Bujaka",MpKrakowWIOSBuja6119,2010-01-01,NaT,tło,miejski,kontenerowa stacjonarna,MAŁOPOLSKIE,Kraków,ul. Bujaka,50.010575,19.949189
297,298,LbLubObywate,PL0507A,"Lublin, ul. Obywatelska",LbLublinWIOS,2010-05-21,NaT,tło,miejski,kontenerowa stacjonarna,LUBELSKIE,Lublin,ul. Obywatelska 13,51.259431,22.569133
298,299,LbLubSliwins,PL0085A,"Lublin, ul. Śliwińskiego",LbLublin_Sliwins_5,2001-07-01,NaT,tło,miejski,kontenerowa stacjonarna,LUBELSKIE,Lublin,ul. Śliwińskiego 5,51.273078,22.551675
1019,1020,WmOlsPuszkin,PL0175A,"Olsztyn, ul. Puszkina",WmOlsztyWIOS_Puszkin,2004-06-01,NaT,tło,miejski,kontenerowa stacjonarna,WARMIŃSKO-MAZURSKIE,Olsztyn,ul. Puszkina 16,53.789233,20.486075


In [53]:
stacje_pm10_pm25 = stacje_pm10[stacje_pm10['Kod stacji'].isin(kody_pm25)]
stacje_pm10_pm25.head()

,Nr,Kod stacji,Kod międzynarodowy,Nazwa stacji,Stary Kod stacji \n(o ile inny od aktualnego),Data uruchomienia,Data zamknięcia,Typ stacji,Typ obszaru,Rodzaj stacji,Województwo,Miejscowość,Adres,WGS84 φ N,WGS84 λ E
182,183,KpBydWarszaw,PL0064A,"Bydgoszcz, ul. Warszawska",KpBydgWarszawska,1995-01-21,NaT,tło,miejski,kontenerowa stacjonarna,KUJAWSKO-POMORSKIE,Bydgoszcz,ul. Warszawska 10,53.134083,17.995708
297,298,LbLubObywate,PL0507A,"Lublin, ul. Obywatelska",LbLublinWIOS,2010-05-21,NaT,tło,miejski,kontenerowa stacjonarna,LUBELSKIE,Lublin,ul. Obywatelska 13,51.259431,22.569133
298,299,LbLubSliwins,PL0085A,"Lublin, ul. Śliwińskiego",LbLublin_Sliwins_5,2001-07-01,NaT,tło,miejski,kontenerowa stacjonarna,LUBELSKIE,Lublin,ul. Śliwińskiego 5,51.273078,22.551675
334,335,LdLodzCzerni,PL0096A,"Łódź, ul. Czernika",LdLodzWIOSACzernik,2003-11-01,NaT,tło,miejski,kontenerowa stacjonarna,ŁÓDZKIE,Łódź,ul. Czernika 1/3,51.758050,19.529786
335,336,LdLodzGdansk,PL0543A,"Łódź, ul. Gdańska",LdLodzWIOSAGdanska,2012-01-01,NaT,tło,miejski,kontenerowa stacjonarna,ŁÓDZKIE,Łódź,ul. Gdańska 16,51.775378,19.450992


In [54]:
print(stacje_pm10["Miejscowość"].value_counts())

Miejscowość
Warszawa     4
Łódź         3
Gdańsk       3
Lublin       2
Bydgoszcz    1
Kraków       1
Białystok    1
Rzeszów      1
Katowice     1
Olsztyn      1
Szczecin     1
Name: count, dtype: int64


In [55]:
print(stacje_pm10_pm25["Miejscowość"].value_counts())

Miejscowość
Łódź         3
Warszawa     3
Gdańsk       3
Lublin       2
Bydgoszcz    1
Kraków       1
Białystok    1
Rzeszów      1
Katowice     1
Olsztyn      1
Szczecin     1
Name: count, dtype: int64


In [56]:
kody = stacje_pm10_pm25['Kod stacji']

stanowiska[
    stanowiska["Kod stacji"].isin(kody)
    & stanowiska["Wskaźnik - kod"].isin(["PM10", "PM2.5"])
    & (stanowiska["Czas uśredniania"] == "1-godzinny")
][["Kod stacji", "Wskaźnik - kod", "Data uruchomienia", "Data zamknięcia"]].sort_values("Kod stacji")

,Kod stacji,Wskaźnik - kod,Data uruchomienia,Data zamknięcia
912,KpBydWarszaw,PM10,1995-01-21 00:00:00,NaN
914,KpBydWarszaw,PM2.5,2007-02-01 00:00:00,NaN
1589,LbLubObywate,PM10,2010-05-21 00:00:00,NaN
1590,LbLubObywate,PM2.5,2015-01-01 00:00:00,NaN
1754,LdLodzCzerni,PM10,2003-11-01 00:00:00,NaN
1756,LdLodzCzerni,PM2.5,2009-06-12 00:00:00,NaN
1765,LdLodzGdansk,PM10,2012-01-01 00:00:00,NaN
1766,LdLodzGdansk,PM2.5,2015-01-29 00:00:00,2019-12-27 00:00:00
2358,MpKrakBujaka,PM10,2010-01-01 00:00:00,NaN
2360,MpKrakBujaka,PM2.5,2010-01-01 00:00:00,2019-12-31 00:00:00


In [57]:
pm10_1g = stanowiska[(stanowiska['Wskaźnik - kod'] == 'PM10') & (stanowiska['Czas uśredniania'] == '1-godzinny')]

pm10_1g['Data zamknięcia'] = pm10_1g["Data zamknięcia"].replace("31.12.20025", "31.12.2025")

start = pd.to_datetime(pm10_1g['Data uruchomienia'], dayfirst=True)
end = pd.to_datetime(pm10_1g['Data zamknięcia'], dayfirst=True).fillna(pd.Timestamp("2100-01-01"))


kody_finalne = pm10_1g.loc[(start <= '2015-01-01') & (end >= '2025-01-01'), "Kod stacji"]
finalne = stacje_pm10[stacje_pm10["Kod stacji"].isin(kody_finalne)]
print(finalne["Miejscowość"].value_counts())

Miejscowość
Warszawa     3
Łódź         2
Gdańsk       2
Bydgoszcz    1
Lublin       1
Kraków       1
Białystok    1
Rzeszów      1
Katowice     1
Olsztyn      1
Szczecin     1
Name: count, dtype: int64


In [58]:
print(finalne.columns.tolist())

['Nr', 'Kod stacji', 'Kod międzynarodowy', 'Nazwa stacji', 'Stary Kod stacji \n(o ile inny od aktualnego)', 'Data uruchomienia', 'Data zamknięcia', 'Typ stacji', 'Typ obszaru', 'Rodzaj stacji', 'Województwo', 'Miejscowość', 'Adres', 'WGS84 φ N', 'WGS84 λ E']


In [59]:
stations = finalne.rename(columns={
    "Kod stacji": "station_code",
    "Stary Kod stacji \n(o ile inny od aktualnego)": "old_code",
    "Kod międzynarodowy": "intl_code",
    "Nazwa stacji": "name",
    "Miejscowość": "city",
    "Typ stacji": "station_type",
    "Typ obszaru": "area_type",
    "Data uruchomienia": "active_from",
    "Data zamknięcia": "active_to",
    "WGS84 φ N": "lat",
    "WGS84 λ E": "lon",
})

kolumny = ["station_code", "old_code", "intl_code", "name", "city",
           "station_type", "area_type", "active_from", "active_to", "lat", "lon"]

stations[kolumny].to_csv("../data/reference/stations.csv", index=False)

In [60]:
raw = pd.read_excel('../data/raw/gios/2019/2019_PM10_1g.xlsx', header=None)

first = raw.iloc[:,0]
row = first[first == 'Kod stacji'].index[0]

print(row)

1


In [61]:
raw1 = pd.read_excel('../data/raw/gios/2015/2015_PM10_1g.xlsx',header = None)

first1 = raw1.iloc[:,0]
row1 = first1[first1 == 'Kod stacji'].index[0]
print(row1)

0


In [62]:
codes = raw.iloc[row,1:].tolist()
print(len(codes))
print(codes[:5])

139
['DsDusznikMOB', 'DsDzialoszyn', 'DsDziePilsud', 'DsJaworMOB', 'DsJelGorOgin']


In [63]:
codes = raw1.iloc[row,1:].tolist()
print(len(codes))
print(codes[:5])

113
['PM10', 'PM10', 'PM10', 'PM10', 'PM10']


In [64]:
raw = pd.read_excel('../data/raw/gios/2015/2015_PM10_1g.xlsx',skiprows = [1,2],header = None)

first = raw.iloc[:,0]
row = first[first == 'Kod stacji'].index[0]
print(row)

0


In [65]:
raw.head()

,0,1,2,3,4,5,6,7,8,9,...,104,105,106,107,108,109,110,111,112,113
0,Kod stacji,DsBogatFrancMOB,DsDzialoszyn,DsDziePilsud,DsJelGorOgin,DsKlodzSzkol,DsLegAlRzecz,DsNowRudSreb,DsSwidnMarciMOB,DsWalbrzWyso,...,WpBoroDrapal,WpKaliSawick,WpKoniWyszyn,WpPilaKusoci,WpPoznDabrow,WpPoznPolank,ZpKoszArKraj,ZpSzczAndr01,ZpSzczecPrze,ZpSzczLacz04
1,2015-01-01 01:00:00,NaN,NaN,85,NaN,54,124,112.047,NaN,37,...,NaN,28.3,32.1,38.9,25.4,35.729,NaN,NaN,NaN,NaN
2,2015-01-01 02:00:00,NaN,NaN,25.5908,307.006,51.3939,18.0403,89.7769,NaN,29.3472,...,NaN,36.4,28.2,142.9,93,37.75,22.7639,117.682,51.7547,70.3454
3,2015-01-01 03:00:00.005000,NaN,NaN,15.0527,264.503,62.7521,21.6418,47.1968,NaN,31.9049,...,NaN,26.1,30.1,36,32.1,46.3137,18.8447,37.2053,18.0278,30.1384
4,2015-01-01 04:00:00.010000,NaN,NaN,13.4938,245.907,59.5961,25.178,37.6718,NaN,33.6525,...,NaN,24.6,31.1,15.8,23,49.34,20.3953,29.7189,17.8709,33.3984


In [66]:
daty = pd.to_datetime(raw[0],errors='coerce', format="mixed")
data = raw[daty.notna()]

data.columns = ["ts"] + codes
data.head()

,ts,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10,...,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10
1,2015-01-01 01:00:00,NaN,NaN,85,NaN,54,124,112.047,NaN,37,...,NaN,28.3,32.1,38.9,25.4,35.729,NaN,NaN,NaN,NaN
2,2015-01-01 02:00:00,NaN,NaN,25.5908,307.006,51.3939,18.0403,89.7769,NaN,29.3472,...,NaN,36.4,28.2,142.9,93,37.75,22.7639,117.682,51.7547,70.3454
3,2015-01-01 03:00:00.005000,NaN,NaN,15.0527,264.503,62.7521,21.6418,47.1968,NaN,31.9049,...,NaN,26.1,30.1,36,32.1,46.3137,18.8447,37.2053,18.0278,30.1384
4,2015-01-01 04:00:00.010000,NaN,NaN,13.4938,245.907,59.5961,25.178,37.6718,NaN,33.6525,...,NaN,24.6,31.1,15.8,23,49.34,20.3953,29.7189,17.8709,33.3984
5,2015-01-01 05:00:00.015000,NaN,NaN,22.2307,217.72,64.223,19.773,26.0654,NaN,34.5415,...,NaN,18.2,22.4,18.2,23.4,36.7124,26.4891,33.9612,23.8245,34.5434


In [67]:
data['ts'] = pd.to_datetime(data['ts']).dt.round("h")
data.head()

,ts,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10,...,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10,PM10
1,2015-01-01 01:00:00,NaN,NaN,85,NaN,54,124,112.047,NaN,37,...,NaN,28.3,32.1,38.9,25.4,35.729,NaN,NaN,NaN,NaN
2,2015-01-01 02:00:00,NaN,NaN,25.5908,307.006,51.3939,18.0403,89.7769,NaN,29.3472,...,NaN,36.4,28.2,142.9,93,37.75,22.7639,117.682,51.7547,70.3454
3,2015-01-01 03:00:00,NaN,NaN,15.0527,264.503,62.7521,21.6418,47.1968,NaN,31.9049,...,NaN,26.1,30.1,36,32.1,46.3137,18.8447,37.2053,18.0278,30.1384
4,2015-01-01 04:00:00,NaN,NaN,13.4938,245.907,59.5961,25.178,37.6718,NaN,33.6525,...,NaN,24.6,31.1,15.8,23,49.34,20.3953,29.7189,17.8709,33.3984
5,2015-01-01 05:00:00,NaN,NaN,22.2307,217.72,64.223,19.773,26.0654,NaN,34.5415,...,NaN,18.2,22.4,18.2,23.4,36.7124,26.4891,33.9612,23.8245,34.5434


In [68]:
long = data.melt(id_vars="ts",var_name="station_code", value_name="value")
print(long.shape)
long.head()

(989880, 3)


,ts,station_code,value
0,2015-01-01 01:00:00,PM10,NaN
1,2015-01-01 02:00:00,PM10,NaN
2,2015-01-01 03:00:00,PM10,NaN
3,2015-01-01 04:00:00,PM10,NaN
4,2015-01-01 05:00:00,PM10,NaN


In [69]:
long["value"] = pd.to_numeric(long["value"], errors="coerce")
long = long.dropna(subset=["value"])
long["pollutant"] = "PM10"

print(long.shape)
long.head()

(922149, 4)


,ts,station_code,value,pollutant
324,2015-01-14 13:00:00,PM10,46.4334,PM10
325,2015-01-14 14:00:00,PM10,33.3692,PM10
326,2015-01-14 15:00:00,PM10,29.7249,PM10
327,2015-01-14 16:00:00,PM10,109.9720,PM10
328,2015-01-14 17:00:00,PM10,99.7845,PM10


In [72]:
for rok in [2015, 2019]:
    df = load_gios_hourly(f"../data/raw/gios/{rok}/{rok}_PM10_1g.xlsx")
    print(rok, df.shape, df["ts"].min(), df["ts"].max())

2015 (922149, 4) 2015-01-01 01:00:00 2016-01-01 00:00:00
2019 (1171407, 4) 2019-01-01 01:00:00 2020-01-01 00:00:00


## Parsowanie archiwum GIOŚ 2015–2025

In [73]:
years =[]
for rok in range(2015,2026):
    df = load_gios_hourly(f"../data/raw/gios/{rok}/{rok}_PM10_1g.xlsx")
    years.append(df)

main = pd.concat(years,ignore_index=True)

In [74]:
print(main.shape)
print(main["ts"].min(), main["ts"].max())
print(main["ts"].dt.year.value_counts().sort_index())

(9321024, 4)
2015-01-01 01:00:00 2025-01-01 00:00:00
ts
2015     922045
2016        104
2019    1171274
2020    1330380
2021    1461577
2022    1422531
2023    1493261
2024    1519690
2025        162
Name: count, dtype: int64


In [75]:
for rok in [2016, 2017, 2018, 2025]:
    df = load_gios_hourly(f"../data/raw/gios/{rok}/{rok}_PM10_1g.xlsx")
    print(rok, df.shape)

2016 (0, 4)
2017 (0, 4)
2018 (0, 4)
2025 (1562519, 4)


In [76]:
raw = pd.read_excel("../data/raw/gios/2016/2016_PM10_1g.xlsx", header=None)

first = raw[0]
row = first[first == "Kod stacji"].index[0]
dates = pd.to_datetime(raw[0], errors="coerce", format="mixed")

print("wiersz z kodami:", row)
print("wiersze z datą:", dates.notna().sum())
print("typ daty:", type(raw.iloc[7, 0]), raw.iloc[7, 0])
print("typ wartości:", type(raw.iloc[7, 1]), raw.iloc[7, 1])

wiersz z kodami: 1
wiersze z datą: 8784
typ daty: <class 'datetime.datetime'> 2016-01-01 02:00:00
typ wartości: <class 'str'> 66,1195


In [77]:
years =[]
for rok in range(2015,2026):
    df = load_gios_hourly(f"../data/raw/gios/{rok}/{rok}_PM10_1g.xlsx")
    years.append(df)

main = pd.concat(years,ignore_index=True)

In [78]:
print(main.shape)
print(main["ts"].min(), main["ts"].max())
print(main["ts"].dt.year.value_counts().sort_index())

(13915333, 4)
2015-01-01 01:00:00 2026-01-01 00:00:00
ts
2015     922045
2016     982775
2017    1020061
2018    1028945
2019    1171387
2020    1330380
2021    1461577
2022    1422531
2023    1493261
2024    1519690
2025    1562511
2026        170
Name: count, dtype: int64


In [79]:
main.head()

,station_code,pollutant,ts,value
0,DsBogatFrancMOB,PM10,2015-01-14 13:00:00,46.4334
1,DsBogatFrancMOB,PM10,2015-01-14 14:00:00,33.3692
2,DsBogatFrancMOB,PM10,2015-01-14 15:00:00,29.7249
3,DsBogatFrancMOB,PM10,2015-01-14 16:00:00,109.9720
4,DsBogatFrancMOB,PM10,2015-01-14 17:00:00,99.7845


In [81]:
stations = pd.read_csv('../data/reference/stations.csv')
stations.head()

,station_code,old_code,intl_code,name,city,station_type,area_type,active_from,active_to,lat,lon
0,KpBydWarszaw,KpBydgWarszawska,PL0064A,"Bydgoszcz, ul. Warszawska",Bydgoszcz,tło,miejski,1995-01-21,NaN,53.134083,17.995708
1,LbLubObywate,LbLublinWIOS,PL0507A,"Lublin, ul. Obywatelska",Lublin,tło,miejski,2010-05-21,NaN,51.259431,22.569133
2,LdLodzCzerni,LdLodzWIOSACzernik,PL0096A,"Łódź, ul. Czernika",Łódź,tło,miejski,2003-11-01,NaN,51.758050,19.529786
3,LdLodzGdansk,LdLodzWIOSAGdanska,PL0543A,"Łódź, ul. Gdańska",Łódź,tło,miejski,2012-01-01,NaN,51.775378,19.450992
4,MpKrakBujaka,MpKrakowWIOSBuja6119,PL0501A,"Kraków, ul. Bujaka",Kraków,tło,miejski,2010-01-01,NaN,50.010575,19.949189


In [89]:
nowe = set(stations["station_code"])
stare = set(stations["old_code"].str.split(", ").explode())

for rok, grupa in main.groupby(main["ts"].dt.year):
    kody = set(grupa["station_code"].unique())
    print(rok, "new:", len(kody & nowe), "old:", len(kody & stare))

2015 new: 14 old: 0
2016 new: 14 old: 0
2017 new: 14 old: 0
2018 new: 15 old: 0
2019 new: 15 old: 0
2020 new: 15 old: 0
2021 new: 14 old: 0
2022 new: 14 old: 0
2023 new: 14 old: 0
2024 new: 15 old: 0
2025 new: 15 old: 0
2026 new: 15 old: 0


In [90]:
for rok, grupa in main.groupby(main["ts"].dt.year):
    kody = set(grupa["station_code"].unique())
    print(rok, "no new:", sorted(nowe - kody), "old:", sorted(kody & stare))

2015 no new: ['MzWarTolstoj'] old: []
2016 no new: ['MzWarTolstoj'] old: []
2017 no new: ['MzWarTolstoj'] old: []
2018 no new: [] old: []
2019 no new: [] old: []
2020 no new: [] old: []
2021 no new: ['MzWarKondrat'] old: []
2022 no new: ['MzWarKondrat'] old: []
2023 no new: ['MzWarKondrat'] old: []
2024 no new: [] old: []
2025 no new: [] old: []
2026 no new: [] old: []


In [91]:
for rok in [2015, 2021]:
    kody = main.loc[main["ts"].dt.year == rok, "station_code"].unique()
    print(rok, sorted(k for k in kody if k.startswith("MzWar")))

2015 ['MzWarKondrat', 'MzWarWokalna']
2021 ['MzWarTolstoj', 'MzWarWokalna']


In [92]:
pary = stations[["station_code", "old_code"]].copy()
pary["old_code"] = pary["old_code"].str.split(", ")
pary = pary.explode("old_code")
mapa = dict(zip(pary["old_code"], pary["station_code"]))

main["station_code"] = main["station_code"].replace(mapa)
main = main[main["station_code"].isin(nowe)]

print(main.shape)
print(main.duplicated(["station_code", "ts"]).sum())

(1329333, 4)
0


In [93]:
for rok, grupa in main.groupby(main["ts"].dt.year):
    kody = set(grupa["station_code"].unique())
    print(rok, "new:", len(kody & nowe), "old:", len(kody & stare))

2015 new: 14 old: 0
2016 new: 14 old: 0
2017 new: 14 old: 0
2018 new: 15 old: 0
2019 new: 15 old: 0
2020 new: 15 old: 0
2021 new: 14 old: 0
2022 new: 14 old: 0
2023 new: 14 old: 0
2024 new: 15 old: 0
2025 new: 15 old: 0
2026 new: 15 old: 0


In [95]:
negative = (main["value"] < 0).sum()
print(negative)
main = main[main["value"] >= 0]

0


In [97]:
completeness = main.groupby(['station_code', main['ts'].dt.year]).size().unstack()
completeness

ts,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
station_code,,,,,,,,,,,,
KpBydWarszaw,8345.0,8329.0,8479.0,8477.0,8272.0,7462.0,8590.0,1.0,8681.0,8665.0,8730.0,1.0
LbLubObywate,8427.0,8727.0,8697.0,8638.0,8413.0,8763.0,8729.0,8178.0,8377.0,8753.0,8539.0,1.0
LdLodzCzerni,8167.0,7547.0,8176.0,8663.0,8645.0,8679.0,8678.0,8696.0,8610.0,8285.0,8681.0,1.0
LdLodzGdansk,8692.0,8239.0,8610.0,8684.0,8661.0,8722.0,8693.0,1.0,7337.0,8588.0,8662.0,1.0
MpKrakBujaka,8702.0,8633.0,8260.0,8357.0,8655.0,8603.0,8687.0,8584.0,8382.0,8650.0,8718.0,1.0
MzWarKondrat,8537.0,8139.0,7462.0,8587.0,8709.0,1.0,NaN,NaN,NaN,8639.0,8472.0,1.0
MzWarTolstoj,NaN,NaN,NaN,8251.0,8305.0,8401.0,8756.0,8288.0,8748.0,8415.0,8752.0,1.0
MzWarWokalna,8597.0,8693.0,8571.0,8529.0,8212.0,8502.0,8371.0,8468.0,8515.0,8512.0,8194.0,1.0
PdBialWarsza,8582.0,7457.0,8330.0,7882.0,8619.0,8660.0,8128.0,8497.0,8255.0,8627.0,8457.0,1.0


In [99]:
df_2022 = load_gios_hourly("../data/raw/gios/2022/2022_PM10_1g.xlsx")
codes_2022 = df_2022["station_code"].unique()
print(sorted(c for c in codes_2022 if c.startswith(("KpByd", "LdLodz"))))

df_2020 = load_gios_hourly("../data/raw/gios/2020/2020_PM10_1g.xlsx")
codes_2020 = df_2020["station_code"].unique()
print(sorted(c for c in codes_2020 if c.startswith("MzWar")))

['KpBydPlPozna', 'LdLodzCzerni', 'LdLodzJanPaw']
['MzWarAlNiepo', 'MzWarBajkowa', 'MzWarChrosci', 'MzWarTolstoj', 'MzWarWokalna']


In [103]:
main.to_parquet("../data/staging/gios_hourly.parquet", index=False)

## IMGW


In [106]:
s_d = pd.read_csv('../data/raw/imgw/2015/s_d_295_2015.csv',encoding='cp1250',header=None)
s_d.head()

,0,1,2,3,4,5,6,7,8,9,...,55,56,57,58,59,60,61,62,63,64
0,353230295,BIAŁYSTOK,2015,1,1,1.9,NaN,-3.2,NaN,0.6,...,NaN,0,NaN,Z,20,NaN,0,9.0,0,8
1,353230295,BIAŁYSTOK,2015,1,2,4.5,NaN,0.2,NaN,2.4,...,NaN,0,NaN,R,15,NaN,5,NaN,0,8
2,353230295,BIAŁYSTOK,2015,1,3,4.6,NaN,1.3,NaN,2.9,...,NaN,0,NaN,R,0,9.0,0,9.0,0,8
3,353230295,BIAŁYSTOK,2015,1,4,2.1,NaN,0.1,NaN,1.1,...,NaN,0,NaN,R,13,NaN,5,NaN,0,8
4,353230295,BIAŁYSTOK,2015,1,5,1.3,NaN,-1.8,NaN,-0.4,...,NaN,0,NaN,R,12,NaN,4,NaN,0,8


In [107]:
print(s_d.shape)

(365, 65)


In [108]:
def read_format(path: str) -> list[str]:
    """Return column names from an IMGW format description file."""
    names = []
    with open(path, encoding='cp1250') as f:
        for line in f:
            parts = line.split()
            if parts and parts[0].isupper():
                names.append(parts[0])
    return names

s_d_names = read_format("../data/raw/imgw/s_d_format.txt")
s_d_t_names = read_format("../data/raw/imgw/s_d_t_format.txt")

print(len(s_d_names), len(s_d_t_names))

65 23


In [110]:
s_d.columns = s_d_names
s_d.head()

,NSP,POST,ROK,MC,DZ,TMAX,WTMAX,TMIN,WTMIN,STD,...,WDZPS,DZBL,WDZBL,SGR,IZD,WIZD,IZG,WIZG,AKTN,WAKTN
0,353230295,BIAŁYSTOK,2015,1,1,1.9,NaN,-3.2,NaN,0.6,...,NaN,0,NaN,Z,20,NaN,0,9.0,0,8
1,353230295,BIAŁYSTOK,2015,1,2,4.5,NaN,0.2,NaN,2.4,...,NaN,0,NaN,R,15,NaN,5,NaN,0,8
2,353230295,BIAŁYSTOK,2015,1,3,4.6,NaN,1.3,NaN,2.9,...,NaN,0,NaN,R,0,9.0,0,9.0,0,8
3,353230295,BIAŁYSTOK,2015,1,4,2.1,NaN,0.1,NaN,1.1,...,NaN,0,NaN,R,13,NaN,5,NaN,0,8
4,353230295,BIAŁYSTOK,2015,1,5,1.3,NaN,-1.8,NaN,-0.4,...,NaN,0,NaN,R,12,NaN,4,NaN,0,8


In [114]:
for col in ["WTMAX", "WTMIN", "WSTD", "WSMDB", "WPKSN"]:
    print(col, s_d[col].value_counts(dropna=False).to_dict())

WTMAX {nan: 365}
WTMIN {nan: 365}
WSTD {nan: 365}
WSMDB {nan: 206, 9.0: 159}
WPKSN {9.0: 338, nan: 27}


In [115]:
print(s_d.loc[s_d["WSMDB"] == 9, "SMDB"].value_counts())
print(s_d.loc[s_d["WPKSN"] == 9, "PKSN"].value_counts())

SMDB
0.0    159
Name: count, dtype: int64
PKSN
0    338
Name: count, dtype: int64


In [116]:
s_d_t = pd.read_csv('../data/raw/imgw/2015/s_d_t_295_2015.csv',encoding='cp1250',header=None)
s_d_t.head()

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
0,353230295,BIAŁYSTOK,2015,1,1,8.0,NaN,3.3,NaN,0.6,...,95.3,NaN,1005.7,NaN,1024.7,NaN,0.0,NaN,0.0,NaN
1,353230295,BIAŁYSTOK,2015,1,2,7.4,NaN,5.0,NaN,2.4,...,87.8,NaN,993.2,NaN,1011.8,NaN,2.0,NaN,0.0,NaN
2,353230295,BIAŁYSTOK,2015,1,3,7.4,NaN,6.5,NaN,2.9,...,77.6,NaN,986.8,NaN,1005.3,NaN,0.3,NaN,0.0,NaN
3,353230295,BIAŁYSTOK,2015,1,4,6.9,NaN,4.5,NaN,1.1,...,81.5,NaN,984.9,NaN,1003.4,NaN,0.7,NaN,0.0,NaN
4,353230295,BIAŁYSTOK,2015,1,5,6.1,NaN,4.1,NaN,-0.4,...,73.4,NaN,995.3,NaN,1014.2,NaN,0.6,NaN,0.0,NaN


In [117]:
s_d_t.shape

(365, 23)

In [118]:
s_d_t.columns = s_d_t_names
s_d_t.head()

,NSP,POST,ROK,MC,DZ,NOS,WNOS,FWS,WFWS,TEMP,...,WLGS,WWLGS,PPPS,WPPPS,PPPM,WPPPM,WODZ,WWODZ,WONO,WWONO
0,353230295,BIAŁYSTOK,2015,1,1,8.0,NaN,3.3,NaN,0.6,...,95.3,NaN,1005.7,NaN,1024.7,NaN,0.0,NaN,0.0,NaN
1,353230295,BIAŁYSTOK,2015,1,2,7.4,NaN,5.0,NaN,2.4,...,87.8,NaN,993.2,NaN,1011.8,NaN,2.0,NaN,0.0,NaN
2,353230295,BIAŁYSTOK,2015,1,3,7.4,NaN,6.5,NaN,2.9,...,77.6,NaN,986.8,NaN,1005.3,NaN,0.3,NaN,0.0,NaN
3,353230295,BIAŁYSTOK,2015,1,4,6.9,NaN,4.5,NaN,1.1,...,81.5,NaN,984.9,NaN,1003.4,NaN,0.7,NaN,0.0,NaN
4,353230295,BIAŁYSTOK,2015,1,5,6.1,NaN,4.1,NaN,-0.4,...,73.4,NaN,995.3,NaN,1014.2,NaN,0.6,NaN,0.0,NaN


In [119]:
for col in ["WFWS", "WWLGS"]:
    print(col, s_d_t[col].value_counts(dropna=False).to_dict())

WFWS {nan: 365}
WWLGS {nan: 365}


In [139]:
import numpy as np
S_D_VALUES = {"TMAX": "WTMAX", "TMIN": "WTMIN", "STD": "WSTD", "SMDB": "WSMDB", "PKSN": "WPKSN"}
S_D_T_VALUES = {"FWS": "WFWS", "WLGS": "WWLGS"}
KEYS = ["NSP", "ROK", "MC", "DZ"]

RENAME = {
    "NSP": "weather_station_code",
    "STD": "t_mean",
    "TMIN": "t_min",
    "TMAX": "t_max",
    "FWS": "wind_mean",
    "SMDB": "precip",
    "PKSN": "snow_depth",
    "WLGS": "humidity",
}

def read_synop_daily(s_d_path, s_d_t_path) -> pd.DataFrame:
    """Read one station-year of IMGW daily synoptic data (main + additional file).
    If the additional file is missing, wind and humidity are left empty."""
    s_d = pd.read_csv(s_d_path, encoding="cp1250", header=None, names=s_d_names)
    for val, status in S_D_VALUES.items():
        s_d.loc[s_d[status] == 8, val] = np.nan
    s_d = s_d[KEYS + list(S_D_VALUES)]

    if Path(s_d_t_path).exists():
        s_d_t = pd.read_csv(s_d_t_path, encoding="cp1250", header=None, names=s_d_t_names)
        for val, status in S_D_T_VALUES.items():
            s_d_t.loc[s_d_t[status] == 8, val] = np.nan
        df = s_d.merge(s_d_t[KEYS + list(S_D_T_VALUES)], on=KEYS, how="left")
    else:
        df = s_d.copy()
        for val in S_D_T_VALUES:
            df[val] = np.nan

    df['day'] = pd.to_datetime({'year':df['ROK'],'month':df['MC'],'day':df['DZ']})
    df = df.rename(columns = RENAME)
    df["weather_station_code"] = df["weather_station_code"].astype(str)
    return df[["weather_station_code", "day", "t_mean", "t_min", "t_max", "wind_mean", "precip", "snow_depth", "humidity"]]

In [140]:
test = read_synop_daily(
    "../data/raw/imgw/2015/s_d_295_2015.csv",
    "../data/raw/imgw/2015/s_d_t_295_2015.csv",
)
print(test.shape)
test.head()

(365, 9)


,weather_station_code,day,t_mean,t_min,t_max,wind_mean,precip,snow_depth,humidity
0,353230295,2015-01-01,0.6,-3.2,1.9,3.3,0.0,3.0,95.3
1,353230295,2015-01-02,2.4,0.2,4.5,5.0,2.0,2.0,87.8
2,353230295,2015-01-03,2.9,1.3,4.6,6.5,0.3,0.0,77.6
3,353230295,2015-01-04,1.1,0.1,2.1,4.5,0.7,0.0,81.5
4,353230295,2015-01-05,-0.4,-1.8,1.3,4.1,0.6,1.0,73.4


In [141]:
from pathlib import Path
def build_imgw_staging(raw_dir: Path = Path("../data/raw/imgw"),stations_path: Path = Path("../data/reference/weather_stations.csv"), output_path: Path = Path("../data/staging/imgw_daily.parquet"),) -> pd.DataFrame:
    """Parse daily synoptic data for all selected IMGW stations and years and save the result to Parquet."""
    stations = pd.read_csv(stations_path, dtype = str)
    file_codes = stations['file_code']

    frames =[]

    for year in range(2015,2026):
        for code in file_codes:
            s_d_path = raw_dir / str(year) / f's_d_{code}_{year}.csv'
            s_d_t_path = raw_dir / str(year) / f's_d_t_{code}_{year}.csv'
            frames.append(read_synop_daily(s_d_path, s_d_t_path))

    weather = pd.concat(frames, ignore_index=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    weather.to_parquet(output_path, index=False)
    return weather


In [142]:
weather = build_imgw_staging()
print(weather.shape)
print(weather["weather_station_code"].nunique())

(44194, 9)
11


In [147]:
print(weather["day"].dt.year.value_counts().sort_index())

day
2015    4015
2016    4026
2017    4015
2018    4015
2019    4015
2020    4026
2021    4015
2022    4015
2023    4015
2024    4022
2025    4015
Name: count, dtype: int64


In [146]:
y2024 = weather[weather["day"].dt.year == 2024]
print(y2024["weather_station_code"].value_counts())

weather_station_code
353230295    366
350190560    366
350190566    366
351220495    366
351190465    366
353200272    366
350220580    366
353140205    366
352200375    366
353180250    364
354180155    364
Name: count, dtype: int64


In [145]:
all_days = pd.date_range("2024-01-01", "2024-12-31")
for code in ["353180250", "354180155"]:
    days = y2024.loc[y2024["weather_station_code"] == code, "day"]
    print(code, all_days.difference(days).date.tolist())

353180250 [datetime.date(2024, 10, 23), datetime.date(2024, 10, 24)]
354180155 [datetime.date(2024, 10, 23), datetime.date(2024, 10, 24)]


In [148]:
print(weather.groupby(weather["day"].dt.year)[["wind_mean", "humidity"]].count())

      wind_mean  humidity
day                      
2015       4014      4013
2016       4025      4026
2017       4014      4014
2018       4015      4014
2019       4014      4015
2020       4026      4026
2021       4011      4015
2022       3983      4015
2023       4000      4009
2024       2001      2002
2025          0         0


In [149]:
y2024 = weather[weather["day"].dt.year == 2024]
print(y2024.groupby(y2024["day"].dt.month)["wind_mean"].count())

day
1     341
2     319
3     341
4     330
5     341
6     329
7       0
8       0
9       0
10      0
11      0
12      0
Name: wind_mean, dtype: int64


In [150]:
s_t = pd.read_csv('../data/raw/imgw_hourly/2025/s_t_295_2025.csv',encoding="cp1250",header=None)
s_t.head()

,0,1,2,3,4,5,6,7,8,9,...,97,98,99,100,101,102,103,104,105,106
0,353230295,BIAŁYSTOK,2025,1,1,0,NaN,8.0,NaN,8.0,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0
1,353230295,BIAŁYSTOK,2025,1,1,1,NaN,8.0,NaN,8.0,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0
2,353230295,BIAŁYSTOK,2025,1,1,2,NaN,8.0,NaN,8.0,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0
3,353230295,BIAŁYSTOK,2025,1,1,3,4.0,NaN,540.0,NaN,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0
4,353230295,BIAŁYSTOK,2025,1,1,4,4.0,NaN,420.0,NaN,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0


In [151]:
s_t.shape

(8759, 107)

In [152]:
s_t_names = read_format('../data/raw/imgw/s_t_format.txt')

s_t.columns = s_t_names
s_t.head()

,NSP,POST,ROK,MC,DZ,GG,HPOD,WHPOD,HPON,WHPON,...,HSS,WHSS,GRSN,WGRSN,GATS,UKPO,HPRO,WHPRO,CIPR,WCIPR
0,353230295,BIAŁYSTOK,2025,1,1,0,NaN,8.0,NaN,8.0,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0
1,353230295,BIAŁYSTOK,2025,1,1,1,NaN,8.0,NaN,8.0,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0
2,353230295,BIAŁYSTOK,2025,1,1,2,NaN,8.0,NaN,8.0,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0
3,353230295,BIAŁYSTOK,2025,1,1,3,4.0,NaN,540.0,NaN,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0
4,353230295,BIAŁYSTOK,2025,1,1,4,4.0,NaN,420.0,NaN,...,NaN,8.0,NaN,8.0,NaN,NaN,NaN,8.0,NaN,8.0


In [153]:
hours_per_day = s_t.groupby(["MC", "DZ"]).size()
print(hours_per_day.value_counts())

24    364
23      1
Name: count, dtype: int64


In [154]:
for col in ["WFWR", "WWLGW"]:
    print(col, s_t[col].value_counts(dropna=False).to_dict())

WFWR {nan: 8759}
WWLGW {nan: 8759}


In [158]:
S_T_VALUES = {
    'FWR':'WFWR',
    'WLGW':'WWLGW',
}

def read_synop_hourly(s_t_path: Path) -> pd.DataFrame:
    """Read one station-year of IMGW hourly synoptic data and return daily means
    of wind speed and humidity (only days with at least 18 valid hours)."""
    s_t = pd.read_csv(s_t_path, encoding="cp1250", header=None, names=s_t_names)

    for val, status in S_T_VALUES.items():
        s_t.loc[s_t[status] == 8, val] = np.nan

    daily = s_t.groupby(["NSP", "ROK", "MC", "DZ"]).agg(
        wind_mean=("FWR", "mean"),
        wind_n=("FWR", "count"),
        humidity=("WLGW", "mean"),
        humidity_n=("WLGW", "count"),
    ).reset_index()

    daily.loc[daily["wind_n"] < 18, "wind_mean"] = np.nan
    daily.loc[daily["humidity_n"] < 18, "humidity"] = np.nan

    daily["day"] = pd.to_datetime({"year": daily["ROK"], "month": daily["MC"], "day": daily["DZ"]})
    daily = daily.rename(columns={"NSP": "weather_station_code"})
    daily["weather_station_code"] = daily["weather_station_code"].astype(str)
    return daily[["weather_station_code", "day", "wind_mean", "humidity"]]

In [159]:
hourly_test = read_synop_hourly(Path("../data/raw/imgw_hourly/2025/s_t_295_2025.csv"))
print(hourly_test.shape)
hourly_test.describe()

(365, 4)


,day,wind_mean,humidity
count,365,365.000000,365.000000
mean,2025-07-02 00:00:00,2.099623,77.470786
min,2025-01-01 00:00:00,0.583333,40.750000
25%,2025-04-02 00:00:00,1.541667,67.375000
50%,2025-07-02 00:00:00,2.000000,77.875000
75%,2025-10-01 00:00:00,2.541667,88.166667
max,2025-12-31 00:00:00,4.625000,100.000000
std,NaN,0.749691,12.977267


In [160]:
hourly_2024 = read_synop_hourly(Path("../data/raw/imgw_hourly/2024/s_t_295_2024.csv"))

official = weather[
    (weather["weather_station_code"] == "353230295")
    & (weather["day"] < "2024-07-01")
    & (weather["day"] >= "2024-01-01")
]

compare = official.merge(hourly_2024, on=["weather_station_code", "day"], suffixes=("_imgw", "_hourly"))

print(len(compare))
print((compare["wind_mean_imgw"] - compare["wind_mean_hourly"]).abs().mean())
print((compare["humidity_imgw"] - compare["humidity_hourly"]).abs().mean())

182
0.12062231247013853
0.611868928173276


In [161]:
weather = pd.read_parquet("../data/staging/imgw_daily.parquet")
print(weather.groupby(weather["day"].dt.year)[["wind_mean", "humidity"]].count())

      wind_mean  humidity
day                      
2015       4014      4013
2016       4025      4026
2017       4014      4014
2018       4015      4014
2019       4014      4015
2020       4026      4026
2021       4011      4015
2022       3983      4015
2023       4000      4009
2024       4022      4022
2025       4015      4015


In [162]:
weather.describe()

,day,t_mean,t_min,t_max,wind_mean,precip,snow_depth,humidity
count,44194,44192.000000,44193.000000,44193.000000,44139.000000,40632.000000,35885.000000,44184.000000
mean,2020-07-01 08:34:43.395936,9.731128,5.519573,14.274747,3.059767,1.759625,0.690929,76.920847
min,2015-01-01 00:00:00,-19.700000,-26.000000,-15.700000,0.100000,0.000000,0.000000,28.900000
25%,2017-10-01 00:00:00,3.400000,0.200000,6.500000,2.000000,0.000000,0.000000,68.300000
50%,2020-07-01 00:00:00,9.400000,5.300000,14.200000,2.800000,0.000000,0.000000,78.375000
75%,2023-04-02 00:00:00,16.500000,11.300000,22.200000,3.800000,1.500000,0.000000,87.000000
max,2025-12-31 00:00:00,29.300000,24.000000,36.700000,12.375000,140.700000,46.000000,100.300000
std,NaN,7.974629,7.128438,9.392278,1.414551,4.489426,2.890011,12.700233


In [163]:
missing = weather.assign(year=weather["day"].dt.year).groupby(["weather_station_code", "year"])[["precip", "snow_depth"]].apply(lambda g: g.isna().sum())
print(missing[(missing["precip"] > 0) | (missing["snow_depth"] > 0)])

                           precip  snow_depth
weather_station_code year                    
350190560            2024     150         339
                     2025     164         319
350190566            2024     171         343
                     2025     170         333
350220580            2024     173         344
                     2025     153         336
351190465            2024     154         350
                     2025     157         336
351220495            2024     177         336
                     2025     160         323
352200375            2024     165         342
                     2025     170         344
353140205            2024     157         352
                     2025     198         349
353180250            2024     149         342
                     2025     173         335
353200272            2015       0         365
                     2016       0         366
                     2017       0          89
                     2024     147 

In [164]:
print(weather.loc[weather["precip"].idxmax()])
print((weather["humidity"] > 100).sum())

weather_station_code              350190560
day                     2024-09-14 00:00:00
t_mean                                 15.8
t_min                                  14.2
t_max                                  17.4
wind_mean                          2.208333
precip                                140.7
snow_depth                              NaN
humidity                              96.75
Name: 37508, dtype: object
2


In [165]:
raw_2024 = pd.read_csv("../data/raw/imgw/2024/s_d_295_2024.csv", header=None, encoding="cp1250", names=s_d_names)
print(raw_2024.loc[raw_2024["WSMDB"] == 9, "SMDB"].value_counts(dropna=False))

SMDB
NaN    165
Name: count, dtype: int64


In [170]:
weather = pd.read_parquet("../data/staging/imgw_daily.parquet")
missing = weather.assign(year=weather["day"].dt.year).groupby(["weather_station_code", "year"])[["precip", "snow_depth"]].apply(lambda g: g.isna().sum())
print(missing[(missing["precip"] > 0) | (missing["snow_depth"] > 0)])

                           precip  snow_depth
weather_station_code year                    
353200272            2015       0         365
                     2016       0         366
                     2017       0          89
                     2025       0           2
353230295            2022       0           1
354180155            2023       0          38
                     2024       0         180
                     2025       0         159


In [171]:
swibno = weather[(weather["weather_station_code"] == "354180155") & (weather["day"].dt.year == 2024)]
print(swibno.groupby(swibno["day"].dt.month)["snow_depth"].apply(lambda s: s.isna().sum()))

day
1     16
2      8
3     19
4      9
5     21
6     13
7     18
8     16
9     10
10    13
11    25
12    12
Name: snow_depth, dtype: int64
